# Slow-roll inflation and the (n_s, r) plane

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/inflation_slow_roll)

Five inflaton potentials ($m^2\phi^2$, $\lambda\phi^4$, Starobinsky $R^2$, natural inflation, a quartic hilltop) roll in one batched tensor through the Friedmann and Klein–Gordon equations, with the number of e-folds $N = \ln a$ as the clock. The slow-roll parameters $\epsilon$, $\eta$ and the predictions $n_s$ and $r$ at $N_* = 50$–$60$ are checked against the exact background and placed against Planck 2018 and BICEP/Keck 2021: $m^2\phi^2$ and $\lambda\phi^4$ are disfavoured, Starobinsky sits inside. Inflation is the leading explanation of the initial conditions, but it is not confirmed: primordial B-modes have not been detected.

**Vector form**

$$H^2 = \frac{\dot\phi^2/2 + V}{3M_\mathrm{pl}^2},\qquad \ddot\phi + 3H\dot\phi + V'(\phi) = 0,\qquad \epsilon_V = \frac{M_\mathrm{pl}^2}{2}\Big(\frac{V'}{V}\Big)^2,\qquad \eta_V = M_\mathrm{pl}^2\frac{V''}{V},\qquad n_s = 1 - 6\epsilon_V + 2\eta_V,\qquad r = 16\,\epsilon_V$$

**Index (tensor) form**

$$\frac{d\phi_b}{dN} = \psi_b,\qquad \frac{d\psi_b}{dN} = -\Big(3 - \frac{\psi_b^2}{2}\Big)\Big(\psi_b + \frac{w_{bp}U'_p(\phi_b)}{w_{bq}U_q(\phi_b)}\Big),\qquad \epsilon_{1,b} = \frac{\psi_b^2}{2},\qquad \epsilon_{i+1} = \frac{d\ln\epsilon_i}{dN}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $w_{bp}$ | `model(KINDS, f, mu, device)` | (B, 5) | one-hot row: field $b$ uses potential $p$ |
| $U_p(\phi_b)$ | `shapes(phi, f, mu)` | (B, 5) | the five potential shapes evaluated on every field |
| $V_b = w_{bp}U_p$ | `einsum('bp,...bp->...b', w, U)` | (B,) | repeated $p$ ⇒ summed: each field's own potential |
| $\phi_b,\ \psi_b$ | `torch.stack([phi, psi])` | (2, B) | field and $d\phi/dN$, advanced together by RK4 in e-folds |
| $V'/V$ | `log_slope(phi, model)` | (B,) | the only thing the background needs: the height of $V$ drops out |
| $\epsilon_V,\ \eta_V$ | `slow_roll_potential(phi, model)` | (B,) | $V'$ and $V''$ by autograd |
| $N_\mathrm{end}$ | `sol['N_end']` | (B,) | where $\epsilon_H$ rises through 1, i.e. $\ddot a$ changes sign |
| $\epsilon_{1,2,3}(N_*)$ | `predictions(sol, model, N_star)` | (S, B) | Hubble-flow parameters $N_*$ e-folds before the end, by one gather |
| $n_s,\ r$ | `ns_r_hubble_flow(e1, e2, e3)` | (S, B) | second-order predictions from the exact background |

Units: reduced Planck mass $M_\mathrm{pl} = 1$. Writing the equations with $N = \ln a$ as time, $\psi = d\phi/dN$ and $\epsilon_H = \psi^2/2$ gives $d\psi/dN = -(3-\epsilon_H)(\psi + V'/V)$ and $H^2 = V/(3-\epsilon_H)$, so only $V'/V$ matters and every potential is drawn with height 1 (the amplitude $A_s = 2.1\times10^{-9}$ then fixes the energy scale, printed as $V_*^{1/4}$, about $10^{16}$ GeV for most models). Every field starts about 68 e-folds before its end, at a point from the textbook slow-roll count, on the slow-roll velocity $\psi = -V'/V$; the test starts $m^2\phi^2$ with very different velocities and finds all of them on the same curve within a few e-folds: slow roll is an attractor. Inflation ends where $\epsilon_H = 1$. The predictions are computed twice: from the potential at first order ($n_s = 1 - 6\epsilon_V + 2\eta_V$, $r = 16\epsilon_V$, the × marks) and from the Hubble-flow parameters $\epsilon_{1,2,3}$ of the numerical background at second order (Stewart and Lyth 1993; Schwarz, Terrero-Escalante and García 2001; the dots); they differ at $O(\epsilon^2)$, a few $10^{-4}$ in $n_s$. A full mode-by-mode Mukhanov–Sasaki solution is not done here. Results at $N_* = 55$: $m^2\phi^2$ gives $n_s \approx 1 - 2/N_*$, $r \approx 8/N_* \approx 0.14$; $\lambda\phi^4$ gives $r \approx 0.29$ and too red an $n_s$; Starobinsky gives $n_s \approx 0.965$, $r \approx 12/N_*^2 \approx 0.004$; natural inflation ($f = 7$) gives $r \approx 0.08$, above the limit; the quartic hilltop ($\mu = 15$) lands inside. The observational region is drawn as a box (Planck 2018 $n_s = 0.9649 \pm 0.0042$ at 68 %, TT,TE,EE+lowE+lensing; BICEP/Keck 2021 $r_{0.05} < 0.036$ at 95 %), not the published two-dimensional contours, which are slightly tilted and depend on the data combination. Honest status: inflation explains the flatness, the horizon problem and the nearly scale-invariant, Gaussian, adiabatic fluctuations, and $n_s < 1$ was predicted before it was measured; but there are rival ideas (bouncing and ekpyrotic models), the inflaton has not been identified, and the smoking gun, primordial B-mode polarisation from gravitational waves, has not been detected (BICEP2's 2014 claim turned out to be Galactic dust). Try $N_*$ between 50 and 60 (it depends on how the universe reheated), or change $f$ and $\mu$: at $N_* = 55$ natural inflation never reaches the box, because lowering $f$ brings $r$ down but makes $n_s$ too red ($f = 5$: $n_s \approx 0.950$, $r \approx 0.04$).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/inflation_slow_roll/main.py
"""Main loop: five potentials → one batched background integration in e-folds → n_s and r → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

N_START = 68.0  # e-folds before the end at which every field starts (slow-roll estimate)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    model = tensors.model(tensors.KINDS, f=params["f"], mu=params["mu"], device=dev)
    phi0 = tensors.start_field(model, N_START)
    sol = solve.integrate(phi0, equation.slow_roll_velocity(phi0, model), model, dN=0.02, N_max=N_START + 25)

    N_star = params["N_star"]
    N_grid = torch.linspace(50.0, 60.0, 41, dtype=torch.float64, device=dev)  # the usual pivot window
    track = solve.predictions(sol, model, N_grid)  # each (41, B)
    pred = solve.predictions(sol, model, torch.tensor([N_star], dtype=torch.float64, device=dev))
    pred = {k: v[0] for k, v in pred.items()}  # (B,)
    ok = tensors.allowed(pred["ns"], pred["r"], sigmas=2)
    scale = solve.energy_scale(pred["eps_V"], tensors.A_S)

    print(f"Planck 2018: n_s = {tensors.NS_MEAN} ± {tensors.NS_SIGMA};  BICEP/Keck 2021: r < {tensors.R_MAX} (95 %)")
    print(
        f"pivot N_* = {N_star:.0f} e-folds before the end;"
        f" natural f = {params['f']}, hilltop μ = {params['mu']} (M_pl = 1)"
    )
    print(f"{'potential':16s} {'N_total':>7s} {'φ_*':>6s} {'ε_V':>8s} {'η_V':>8s} {'n_s(SR)':>8s} {'n_s(num)':>8s}"
          f" {'r(SR)':>7s} {'r(num)':>7s} {'V_*^¼ (GeV)':>11s}  verdict")  # fmt: skip
    for b, kind in enumerate(tensors.KINDS):
        print(
            f"{tensors.LABELS[kind]:16s} {sol['N_end'][b]:7.1f} {pred['phi'][b]:6.2f} {pred['eps_V'][b]:8.5f}"
            f" {pred['eta_V'][b]:+8.5f} {pred['ns_V'][b]:8.4f} {pred['ns'][b]:8.4f} {pred['r_V'][b]:7.4f}"
            f" {pred['r'][b]:7.4f} {scale[b]:11.2e}  {'inside' if ok[b] else 'disfavoured'}"
        )
    print(
        "SR = potential slow-roll (first order);"
        " num = Hubble-flow parameters of the numerical background (second order)"
    )

    view = {
        "N": sol["N"], "phi": sol["phi"], "N_end": sol["N_end"], "phi0": phi0,
        "N_grid": N_grid, "ns_track": track["ns"], "r_track": track["r"],
        "ns": pred["ns"], "r": pred["r"], "ns_V": pred["ns_V"], "r_V": pred["r_V"], "ok": ok,
    }  # fmt: skip
    view = {k: v.cpu().numpy() for k, v in view.items()}
    view["V"] = lambda phi: equation.shapes(
        torch.as_tensor(phi, dtype=torch.float64), params["f"], params["mu"]
    ).numpy()
    return plot.animate(view, N_star, N_START, out_dir / "inflation_slow_roll.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/inflation_slow_roll/equation.py
r"""Physics → tensor translation: a scalar field rolling in an expanding universe, with e-folds N = ln a as clock.

Units: reduced Planck mass M_pl = 1. The Friedmann and Klein–Gordon equations,

    H² = (φ̇²/2 + V) / 3,        φ̈ + 3Hφ̇ + V'(φ) = 0,

become, with ψ = dφ/dN = φ̇/H and the Hubble slow-roll parameter ε_H = −Ḣ/H² = ψ²/2,

    dφ/dN = ψ,        dψ/dN = −(3 − ε_H) (ψ + V'/V),        H² = V / (3 − ε_H).

Only V'/V enters: the overall height of the potential drops out of the background (and of n_s and r), so every
potential is written with height 1. Inflation (ä > 0, i.e. ä/a = H²(1 − ε_H) > 0) lasts while ε_H < 1.

Predictions at the pivot scale, which left the horizon N_* e-folds before the end:

    potential slow roll (first order):  ε_V = (V'/V)²/2,  η_V = V''/V,  n_s = 1 − 6ε_V + 2η_V,  r = 16 ε_V
    Hubble flow on the exact background (second order, Stewart & Lyth 1993; Schwarz et al. 2001):
        ε₁ = ε_H,  ε_{i+1} = d ln ε_i / dN,  C = γ_E + ln 2 − 2 ≈ −0.7296
        n_s = 1 − 2ε₁ − ε₂ − 2ε₁² − (2C + 3) ε₁ε₂ − C ε₂ε₃,      r = 16 ε₁ (1 + C ε₂)

The batch: B fields, one per potential, each picked from the five formulas by a one-hot row w_{bp},

    V_b = w_{bp} U_p(φ_b)        (repeated p summed)
"""

import math

import torch

C_FLOW = 0.5772156649015329 + math.log(2) - 2  # γ_E + ln 2 − 2
STAROBINSKY = math.sqrt(2 / 3)


def shapes(phi: torch.Tensor, f: float, mu: float) -> torch.Tensor:
    r"""U_p(φ) for the five potentials, each with height 1 where that makes sense: (B,) → (B, 5).

    m²φ²: φ²/2      λφ⁴: φ⁴/4      Starobinsky R²: (1 − e^{−√(2/3) φ})²
    natural: 1 + cos(φ/f)          quartic hilltop: 1 − (φ/μ)⁴
    """
    return torch.stack(
        [
            phi**2 / 2,
            phi**4 / 4,
            (1 - torch.exp(-STAROBINSKY * phi)) ** 2,
            1 + torch.cos(phi / f),
            1 - (phi / mu) ** 4,
        ],
        dim=-1,
    )


def potential(phi: torch.Tensor, model: dict) -> torch.Tensor:
    r"""V_b = w_{bp} U_p(φ_b): repeated p summed.   (..., B) → (..., B)"""
    return torch.einsum("bp,...bp->...b", model["w"], shapes(phi, model["f"], model["mu"]))


def slopes(phi: torch.Tensor, f: float, mu: float) -> torch.Tensor:
    """U'_p(φ), the derivatives of shapes() written out by hand (fast inside the time loop): (B,) → (B, 5)."""
    e = torch.exp(-STAROBINSKY * phi)
    return torch.stack(
        [phi, phi**3, 2 * STAROBINSKY * e * (1 - e), -torch.sin(phi / f) / f, -4 * phi**3 / mu**4],
        dim=-1,
    )


def log_slope(phi: torch.Tensor, model: dict) -> torch.Tensor:
    r"""V'/V = w_{bp} U'_p / w_{bp} U_p (repeated p summed).   (..., B) → (..., B)"""
    dV = torch.einsum("bp,...bp->...b", model["w"], slopes(phi, model["f"], model["mu"]))
    return dV / potential(phi, model)


def autograd_slope(phi: torch.Tensor, model: dict) -> torch.Tensor:
    """V'/V by autograd (each V_b depends only on φ_b, so the gradient of the sum is the batch of slopes)."""
    with torch.enable_grad():
        x = phi.detach().requires_grad_(True)
        V = potential(x, model)
        (dV,) = torch.autograd.grad(V.sum(), x)
    return dV / V.detach()


def slow_roll_potential(phi: torch.Tensor, model: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """ε_V = (V'/V)²/2 and η_V = V''/V, with V' and V'' from autograd.   (B,) → (B,), (B,)"""
    with torch.enable_grad():
        x = phi.detach().requires_grad_(True)
        V = potential(x, model)
        (dV,) = torch.autograd.grad(V.sum(), x, create_graph=True)
        (d2V,) = torch.autograd.grad(dV.sum(), x)
    V, dV = V.detach(), dV.detach()
    return 0.5 * (dV / V) ** 2, d2V / V


def slow_roll_velocity(phi: torch.Tensor, model: dict) -> torch.Tensor:
    """The slow-roll attractor dφ/dN = −V'/V (from 3Hφ̇ = −V' and H² = V/3)."""
    return -log_slope(phi, model)


def rhs(state: torch.Tensor, model: dict) -> torch.Tensor:
    """d/dN of state = (φ, ψ) stacked as (2, B): the Klein–Gordon equation in e-folds."""
    phi, psi = state
    eps = 0.5 * psi**2
    return torch.stack([psi, -(3 - eps) * (psi + log_slope(phi, model))])


def epsilon_2(psi: torch.Tensor, phi: torch.Tensor, model: dict) -> torch.Tensor:
    """ε₂ = d ln ε_H / dN = 2 ψ'/ψ, with ψ' taken from the equation of motion (no finite differences)."""
    dpsi = -(3 - 0.5 * psi**2) * (psi + log_slope(phi, model))
    return 2 * dpsi / psi


def ns_r_potential(eps_v: torch.Tensor, eta_v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    return 1 - 6 * eps_v + 2 * eta_v, 16 * eps_v


def ns_r_hubble_flow(e1, e2, e3) -> tuple[torch.Tensor, torch.Tensor]:
    C = C_FLOW
    ns = 1 - 2 * e1 - e2 - 2 * e1**2 - (2 * C + 3) * e1 * e2 - C * e2 * e3
    return ns, 16 * e1 * (1 + C * e2)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/inflation_slow_roll/tensors.py
"""Build the concrete tensors: the batch of potentials, their starting points and the observational bounds.

Units: reduced Planck mass M_pl = 1. Starting points come from the textbook slow-roll e-fold counts (Liddle & Lyth;
Martin, Ringeval & Vennin, "Encyclopaedia Inflationaris", 2014); the exact integration then measures the true N.
"""

import math

import torch

KINDS = ["m2phi2", "phi4", "starobinsky", "natural", "hilltop"]
LABELS = {
    "m2phi2": "m²φ²",
    "phi4": "λφ⁴",
    "starobinsky": "Starobinsky R²",
    "natural": "natural",
    "hilltop": "quartic hilltop",
}

# Observations (pivot k = 0.05 Mpc⁻¹):
NS_MEAN, NS_SIGMA = 0.9649, 0.0042  # Planck 2018 X (TT,TE,EE+lowE+lensing), 68 % (arXiv:1807.06211)
R_MAX = 0.036  # BICEP/Keck 2021 (BK18 + Planck + BAO), 95 % upper limit (arXiv:2110.00483)
A_S = 2.1e-9  # Planck 2018 amplitude of the scalar power spectrum, ln(10¹⁰ A_s) = 3.044


def model(kinds: list[str], f: float, mu: float, device: torch.device) -> dict:
    """One-hot rows w_{bp} (B, 5) picking each field's potential, and the constants f (natural) and μ (hilltop)."""
    index = torch.tensor([KINDS.index(k) for k in kinds], device=device)
    w = torch.nn.functional.one_hot(index, len(KINDS)).to(torch.float64)
    return {"kinds": list(kinds), "w": w, "f": f, "mu": mu}


def start_field(m: dict, N_start: float) -> torch.Tensor:
    """φ_i (B,) roughly N_start e-folds before the end of inflation, from the slow-roll count N(φ) of each potential."""
    f, mu, N = m["f"], m["mu"], N_start
    s = math.sqrt(2 / 3)
    # natural: ε_V = tan²(φ/2f)/(2f²) = 1 at the end, N = 2f² ln[sin(φ_e/2f) / sin(φ/2f)]
    phi_end_nat = 2 * f * math.atan(math.sqrt(2) * f)
    # Starobinsky: N = (3/4)[e^{sφ} − e^{sφ_e} − s(φ − φ_e)], ends at φ_e = √(3/2) ln(1 + 2/√3)
    phi_end_star = math.log(1 + 2 / math.sqrt(3)) / s
    phi_star = phi_end_star
    for _ in range(30):  # fixed-point iteration e^{sφ} = 4N/3 + e^{sφ_e} + s(φ − φ_e)
        phi_star = math.log(4 * N / 3 + math.exp(s * phi_end_star) + s * (phi_star - phi_end_star)) / s
    # quartic hilltop, x = φ/μ: N = (μ²/8)(1/x² + x² − 1/x_e² − x_e²), with ε_V = 1 at √8 x_e³/μ = 1 − x_e⁴
    x_end = 1.0
    for _ in range(60):
        x_end = max(1 - math.sqrt(8) * x_end**3 / mu, 1e-6) ** 0.25
    c = 8 * N / mu**2 + 1 / x_end**2 + x_end**2  # 1/u + u = c with u = x²
    x_hill = math.sqrt((c - math.sqrt(c * c - 4)) / 2)
    table = {
        "m2phi2": math.sqrt(4 * N + 2),  # N = (φ² − φ_e²)/4, φ_e = √2
        "phi4": math.sqrt(8 * N + 8),  # N = (φ² − φ_e²)/8, φ_e = √8
        "starobinsky": phi_star,
        "natural": 2 * f * math.asin(math.sin(phi_end_nat / (2 * f)) * math.exp(-N / (2 * f**2))),
        "hilltop": mu * x_hill,
    }
    return torch.tensor([table[k] for k in m["kinds"]], dtype=torch.float64, device=m["w"].device)


def allowed(ns: torch.Tensor, r: torch.Tensor, sigmas: float = 2) -> torch.Tensor:
    """Inside the Planck n_s band (± sigmas σ) and below the BICEP/Keck r limit (a box, not the published contour)."""
    return ((ns - NS_MEAN).abs() < sigmas * NS_SIGMA) & (r < R_MAX)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/inflation_slow_roll/solve.py
"""Integrate every potential at once in e-folds (RK4 on a (2, B) state), then read off the pivot-scale predictions."""

import math

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from . import equation


def integrate(phi0: torch.Tensor, psi0: torch.Tensor, model: dict, dN: float = 0.02, N_max: float = 95.0) -> dict:
    """Trajectories φ, ψ, ε_H of shape (T, B) on the grid N = 0, dN, 2dN, … Each field stops (is frozen) at the
    step where its ε_H first rises through 1 after inflation began; N_end (B,) is that crossing, interpolated."""
    state = torch.stack([phi0, psi0])  # (2, B)
    B = phi0.shape[0]
    started = 0.5 * psi0**2 < 1  # (B,) inflating already? a kinetic start (ε > 1) begins later
    ended = torch.zeros(B, dtype=torch.bool, device=phi0.device)
    N_end = torch.full((B,), math.nan, dtype=phi0.dtype, device=phi0.device)
    history = [state]
    for t in steps(int(N_max / dN), "e-fold steps"):
        alive = (~ended).to(state.dtype)  # (B,) a finished field stops moving
        new = rk4_step(lambda y, _, alive=alive: equation.rhs(y, model) * alive, state, t * dN, dN)
        eps_old, eps_new = 0.5 * state[1] ** 2, 0.5 * new[1] ** 2
        crossing = started & ~ended & (eps_new >= 1)
        N_end = torch.where(crossing, (t + (1 - eps_old) / (eps_new - eps_old)) * dN, N_end)
        ended = ended | crossing
        started = started | (eps_new < 1)
        state = new
        history.append(state)
        if ended.all():
            break
    traj = torch.stack(history)  # (T, 2, B)
    N = torch.arange(len(traj), dtype=phi0.dtype, device=phi0.device) * dN
    phi, psi = traj[:, 0], traj[:, 1]
    return {"N": N, "phi": phi, "psi": psi, "eps": 0.5 * psi**2, "N_end": N_end, "dN": dN}


def at_e_folds(sol: dict, field: torch.Tensor, N_target: torch.Tensor) -> torch.Tensor:
    """Linear interpolation of field (T, B) at times N_target (S, B) → (S, B), with one gather per neighbour."""
    u = (N_target / sol["dN"]).clamp(0, len(sol["N"]) - 1.000001)
    i0 = u.floor().long()
    frac = u - i0
    lo = torch.gather(field, 0, i0)
    hi = torch.gather(field, 0, i0 + 1)
    return lo + frac * (hi - lo)


def predictions(sol: dict, model: dict, N_star: torch.Tensor) -> dict:
    """n_s and r at N_* e-folds before the end, (S,) values of N_* → every entry (S, B)."""
    N_target = sol["N_end"][None, :] - N_star[:, None]  # (S, B)
    # ε₂ along the whole trajectory (from the equation of motion) and ε₃ = d ln ε₂ / dN by central differences
    e2_traj = equation.epsilon_2(sol["psi"], sol["phi"], model)  # (T, B), the batch broadcast over all times
    e3_traj = torch.gradient(e2_traj, spacing=sol["dN"], dim=0)[0] / e2_traj
    phi = at_e_folds(sol, sol["phi"], N_target)
    psi = at_e_folds(sol, sol["psi"], N_target)
    e1, e2, e3 = 0.5 * psi**2, at_e_folds(sol, e2_traj, N_target), at_e_folds(sol, e3_traj, N_target)
    ns, r = equation.ns_r_hubble_flow(e1, e2, e3)
    eps_v, eta_v = equation.slow_roll_potential(phi.flatten(), _repeat(model, len(N_star)))
    eps_v, eta_v = eps_v.view_as(phi), eta_v.view_as(phi)
    ns_v, r_v = equation.ns_r_potential(eps_v, eta_v)
    return {"phi": phi, "eps_H": e1, "e2": e2, "e3": e3, "ns": ns, "r": r,
            "eps_V": eps_v, "eta_V": eta_v, "ns_V": ns_v, "r_V": r_v}  # fmt: skip


def _repeat(model: dict, S: int) -> dict:
    """The same batch of potentials S times, to evaluate (S, B) field values in one flat call."""
    return {**model, "w": model["w"].repeat(S, 1), "kinds": model["kinds"] * S}


def energy_scale(eps_v: torch.Tensor, A_s: float) -> torch.Tensor:
    """V_*^{1/4} in GeV from the scalar amplitude A_s = V / (24π² ε_V M_pl⁴), with M_pl = 2.435e18 GeV."""
    return (24 * math.pi**2 * eps_v * A_s) ** 0.25 * 2.435e18


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/inflation_slow_roll/plot.py
"""Animate the fields rolling down their potentials (e-fold counter), then their points in the (n_s, r) plane."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import tensors


def animate(view: dict, N_star: float, N_start: float, path: Path) -> Path:
    N, phi, N_end, phi0 = view["N"], view["phi"], view["N_end"], view["phi0"]
    B = len(tensors.KINDS)
    colors = [ACCENT[k] for k in range(B)]
    to_go = np.linspace(N_start, 0.0, 56)  # the clock: e-folds left before each field's end of inflation
    schedule = list(to_go) + [0.0] * 14

    def field_at(left: float) -> np.ndarray:  # φ_b at N_end,b − left, for every field, (B,)
        return np.array([np.interp(N_end[b] - left, N, phi[:, b]) for b in range(B)])

    fig, (ax, ax_p) = plt.subplots(1, 2, figsize=(8.4, 4), gridspec_kw={"width_ratios": [1, 1]})
    phi_end = field_at(0.0)
    curves, balls = [], []
    for b, kind in enumerate(tensors.KINDS):
        lo, hi = sorted([phi0[b], phi_end[b]])
        grid = np.linspace(max(lo - 1.0, 0.0), hi + 1.0, 300)
        V = view["V"](grid)[:, b] / view["V"](np.array([phi0[b]]))[0, b]  # V / V(φ_start)
        curves.append(ax.plot(grid, V, color=colors[b], lw=1.6, label=tensors.LABELS[kind])[0])
        balls.append(ax.plot([], [], "o", ms=8, color=colors[b], mec="#111827", zorder=5)[0])
    ax.set_xlim(0, 1.05 * max(phi0.max(), phi_end.max()) + 1)
    ax.set_ylim(0, 1.12)
    ax.set_xlabel("field φ  (M_pl)")
    ax.set_ylabel("V(φ) / V(φ_start)")
    ax.legend(fontsize=7, loc="lower right")
    clock = ax.text(0.03, 0.97, "", transform=ax.transAxes, va="top", fontsize=9)
    ax.set_title("each field rolls downhill; ● = now", fontsize=9)

    # observations: Planck n_s bands (1σ, 2σ) and the BICEP/Keck upper limit on r
    m, s = tensors.NS_MEAN, tensors.NS_SIGMA
    ax_p.axvspan(m - 2 * s, m + 2 * s, ymax=1, color="#93c5fd", alpha=0.35, lw=0)
    ax_p.axvspan(m - s, m + s, color="#2563eb", alpha=0.25, lw=0, label="Planck 2018 n_s (1σ, 2σ)")
    ax_p.axhspan(tensors.R_MAX, 1.0, color="#fecaca", alpha=0.45, lw=0, label="r > 0.036 excluded (BK18)")
    ax_p.axhline(tensors.R_MAX, color=ACCENT[1], lw=1.2, ls="--")
    ax_p.set_yscale("log")
    ax_p.set_xlim(0.935, 0.985)
    ax_p.set_ylim(1e-3, 0.6)
    ax_p.set_xlabel("spectral index n_s")
    ax_p.set_ylabel("tensor-to-scalar ratio r")
    tracks = [ax_p.plot([], [], "-", color=colors[b], lw=2.5, alpha=0.8)[0] for b in range(B)]
    dots = [ax_p.plot([], [], "o", ms=7, color=colors[b], mec="#111827", zorder=5)[0] for b in range(B)]
    sr = [ax_p.plot([], [], "x", ms=6, color=colors[b], zorder=6)[0] for b in range(B)]
    ax_p.legend(fontsize=7, loc="lower left")
    verdict = ax_p.text(
        0.97,
        0.97,
        "",
        transform=ax_p.transAxes,
        ha="right",
        va="top",
        fontsize=7.5,
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
    )
    ax_p.set_title("predictions at the pivot, N_* = 60 → 50", fontsize=9)
    Ng, ns_t, r_t = view["N_grid"], view["ns_track"], view["r_track"]  # (S,), (S, B), (S, B)

    def update(i: int):
        left = schedule[i]
        now = field_at(left)
        V_now = view["V"](now)[np.arange(B), np.arange(B)] / view["V"](phi0)[np.arange(B), np.arange(B)]
        for b in range(B):
            balls[b].set_data([now[b]], [V_now[b]])
        phase = "inflating" if left > 0 else "inflation over (ε = 1)"
        window = "  ← pivot scales leave the horizon" if 50 <= left <= 60 else ""
        clock.set_text(f"e-folds to go: {left:5.1f}{window}\n{phase}")
        upto = Ng >= max(left, 50.0)  # tracks drawn from N_* = 60 down to the current clock
        for b in range(B):
            if left <= 60:
                tracks[b].set_data(ns_t[upto, b], r_t[upto, b])
                k = int(np.argmin(np.abs(Ng - max(left, 50.0))))
                dots[b].set_data([ns_t[k, b]], [r_t[k, b]])
        if i >= len(to_go):  # final frames: the chosen N_*, slow-roll crosses and the verdict
            for b in range(B):
                dots[b].set_data([view["ns"][b]], [view["r"][b]])
                sr[b].set_data([view["ns_V"][b]], [view["r_V"][b]])
            lines = [
                f"{tensors.LABELS[k]}: {'inside' if view['ok'][b] else 'disfavoured'}"
                for b, k in enumerate(tensors.KINDS)
            ]
            verdict.set_text(f"N_* = {N_star:.0f}   (● numerical, × slow roll)\n" + "\n".join(lines))

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(schedule), 70), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'N_star': 55,
    'f': 7.0,
    'mu': 15.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.inflation_slow_roll")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.inflation_slow_roll.main")
problem_dir = Path("problems/cosmology/inflation_slow_roll")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
